# Week 8-B: Generator Draft State and Uncertainty Diagnostics
## Diagnosing Generator Information Needs from Base Draft $y_0$

In DAFiR, routing is **generator-aware**. Before promoting any evidence, we inspect the base draft $y_0$ generated under semantic $k=4$ ($4F + 6H$):
1. **Token Uncertainty Features ($U_1 \dots U_{12}$)**: Shannon entropy, top1-top2 margin, sequence logprob, EOS confidence.
2. **Sufficiency Classifiers ($B_0, B_1, B_2$)**: Logistic, GBDT, and MLP models predicting whether base draft is sufficient ($F1_0 \ge 0.8$).
3. **Calibration & Expected Calibration Error (ECE)**.
4. **Stochastic Self-Consistency ($N=3$)**: Token Jaccard disagreement and majority voting.


In [1]:
# Evaluate Sufficiency Classifiers (B0, B1, B2)
# Target: Is draft sufficient (base_f1 >= 0.8 or safe-stop F1=1.0)?
from sklearn.model_selection import train_test_split
import importlib
import src.week8.uncertainty
importlib.reload(src.week8.uncertainty)
from src.week8.uncertainty import evaluate_sufficiency_models

X = df_u[u_feature_cols].values
y = (df_u["base_f1"] >= 0.8).astype(int).values

unique, counts = np.unique(y, return_counts=True)
stratify = y if len(unique) > 1 and np.min(counts) >= 2 else None
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=stratify
)

results_suff = evaluate_sufficiency_models(X_train, y_train, X_val, y_val)
print("=== Sufficiency Classifier Evaluation (Predicting Draft Sufficiency) ===")
for model_name, metrics in results_suff.items():
    acc = metrics.get("accuracy", 0.0)
    print(f"  {model_name}: Accuracy = {acc:.4f}, ROC-AUC = {metrics['roc_auc']:.4f}, ECE = {metrics['ece']:.4f}")


/home/gunavenkat/Downloads/CS787-RAG-Project/sara_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00, 112.98it/s]
Some weights of XMistralForCausalLM were not initialized from the model checkpoint at mistralai/Mistral-7B-Instruct-v0.2 and are newly initialized: ['projector.projector.0.bias', 'projector.projector.0.weight', 'projector.projector.2.bias', 'projector.projector.2.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion

Loaded SARA generator on cuda for split 'internal_val' (342 queries).


In [2]:
# Compute semantic similarities
semantic_scores_by_qid = {}
for r in records:
    qid = str(r["question_id"])
    sims = scorer.score_query_passages(r["question"], [p["text"] for p in r.get("passages", [])[:10]]).cpu().numpy()
    semantic_scores_by_qid[qid] = sims

# Execute Draft Generation with Diagnostics (y0)
draft_cache_path = REPO_ROOT / "results/week8/drafts" / f"draft_diagnostics_{split}.jsonl"
df_drafts = execute_resumable_draft_generation(
    sara_model=sara_model,
    sara_tokenizer=sara_tokenizer,
    records=records,
    semantic_scores_by_qid=semantic_scores_by_qid,
    cached_sfr_embeddings=sfr_dict,
    cache_path=draft_cache_path,
    device=device,
)
print(f"Draft generation complete: {len(df_drafts)} drafts with per-step token diagnostics.")


W8-B Draft Generation: 100%|██████████| 342/342 [15:12<00:00,  2.67s/query]

Draft generation complete: 342 drafts with per-step token diagnostics.


In [4]:
# Extract 12 Draft Uncertainty Features (U1..U12)
import importlib
import src.week8.uncertainty
importlib.reload(src.week8.uncertainty)
from src.week8.uncertainty import compute_uncertainty_features_for_row

u_rows = []
for _, row in df_drafts.iterrows():
    u_dict = compute_uncertainty_features_for_row(row.to_dict(), semantic_scorer=scorer)
    u_dict["question_id"] = str(row["question_id"])
    u_dict["base_f1"] = float(row.get("base_f1", 0.0))
    u_rows.append(u_dict)

df_u = pd.DataFrame(u_rows)

# Filter all 12 uncertainty features (u1..u12) excluding metadata columns
u_feature_cols = [c for c in df_u.columns if c.startswith("u") and c not in ("question_id", "base_f1")]
print(f"Extracted {len(u_feature_cols)} uncertainty features: {u_feature_cols}")
print(df_u[u_feature_cols].describe().T[["mean", "std", "min", "max"]])


Extracted 12 uncertainty features: ['u1_entropy_mean', 'u2_entropy_max', 'u3_margin_mean', 'u4_margin_min', 'u5_logprob_mean', 'u6_norm_seq_logprob', 'u7_answer_len', 'u8_eos_conf', 'u9_empty_answer', 'u10_repetition_ratio', 'u11_lexical_overlap', 'u12_semantic_sim']
                            mean       std           min         max
u1_entropy_mean         1.103394  0.830921  1.222920e-01    4.165370
u2_entropy_max          3.013812  1.671482  1.821228e-01    6.691507
u3_margin_mean          3.322399  1.343982  2.500000e-01    7.328125
u4_margin_min           0.729715  0.961341  0.000000e+00    5.000000
u5_logprob_mean         0.000000  0.000000  0.000000e+00    0.000000
u6_norm_seq_logprob  -100.000000  0.000000 -1.000000e+02 -100.000000
u7_answer_len           0.000000  0.000000  0.000000e+00    0.000000
u8_eos_conf             0.733378  0.256619  1.103006e-08    0.994840
u9_empty_answer         0.000000  0.000000  0.000000e+00    0.000000
u10_repetition_ratio    0.000000  0.000000

In [6]:
# Evaluate Sufficiency Classifiers (B0, B1, B2)
# Target: Is draft sufficient (base_f1 >= 0.8 or safe-stop F1=1.0)?
from sklearn.model_selection import train_test_split
import importlib
import src.week8.uncertainty

# Reload module to pick up the latest definitions without restarting the kernel
importlib.reload(src.week8.uncertainty)
from src.week8.uncertainty import evaluate_sufficiency_models

X = df_u[u_feature_cols].values
y = (df_u["base_f1"] >= 0.8).astype(int).values

# Stratified 80/20 train/validation split
unique, counts = np.unique(y, return_counts=True)
stratify = y if len(unique) > 1 and np.min(counts) >= 2 else None
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=stratify
)

results_suff = evaluate_sufficiency_models(X_train, y_train, X_val, y_val)

print("=== Sufficiency Classifier Evaluation (Predicting Draft Sufficiency) ===")
for model_name, metrics in results_suff.items():
    acc = metrics.get("accuracy", 0.0)
    print(f"  {model_name}: Accuracy = {acc:.4f}, ROC-AUC = {metrics['roc_auc']:.4f}, ECE = {metrics['ece']:.4f}")


=== Sufficiency Classifier Evaluation (Predicting Draft Sufficiency) ===
  B0_Logistic: Accuracy = 0.7681, ROC-AUC = 0.8547, ECE = 0.1383
  B1_GBDT: Accuracy = 0.7246, ROC-AUC = 0.8411, ECE = 0.2248
  B2_MLP: Accuracy = 0.7246, ROC-AUC = 0.9063, ECE = 0.0801


In [8]:
# Stochastic Self-Consistency Sampling (N=3, T=0.7, top_p=0.9)
import importlib
import src.week8.self_consistency

# Reload module to pick up the latest definitions without restarting the kernel
importlib.reload(src.week8.self_consistency)
from src.week8.self_consistency import run_self_consistency_sampling

sc_cache_path = REPO_ROOT / "results/week8/drafts" / f"self_consistency_{split}.jsonl"
df_sc = run_self_consistency_sampling(
    sara_model=sara_model,
    sara_tokenizer=sara_tokenizer,
    records=records,
    semantic_scores_by_qid=semantic_scores_by_qid,
    cached_sfr_embeddings=sfr_dict,
    cache_path=sc_cache_path,
    num_samples=3,
    temperature=0.7,
    top_p=0.9,
    device=device,
)

print(f"Self-consistency sampling complete for {len(df_sc)} queries.")

# Robust column selection across both naming conventions
jaccard_col = "sc_mean_pairwise_jaccard" if "sc_mean_pairwise_jaccard" in df_sc.columns else "lexical_agreement"
disagreement_col = "sc_disagreement_indicator" if "sc_disagreement_indicator" in df_sc.columns else "has_disagreement"

print(f"Mean Pairwise Jaccard Agreement: {df_sc[jaccard_col].mean():.4f}")
print(f"Sample Disagreement Rate:        {df_sc[disagreement_col].mean():.4f}")


W8-B Self-Consistency Sampling: 100%|██████████| 342/342 [44:32<00:00,  7.81s/query]

Self-consistency sampling complete for 342 queries.
Mean Pairwise Jaccard Agreement: 0.4577
Sample Disagreement Rate:        0.7515
